In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

DEVICE: cuda


In [2]:
TRAIN_PATH = "/kaggle/input/datasets/emanmuhammed/grad-proj/train_multilabel10_group.csv"
VAL_PATH   = "/kaggle/input/datasets/emanmuhammed/grad-proj/val_multilabel10_group.csv"
TEST_PATH  = "/kaggle/input/datasets/emanmuhammed/grad-proj/test_multilabel10_group.csv"

In [3]:
train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(len(train_df), len(val_df), len(test_df))

93736 19994 20135


In [4]:
DISEASE_COLS = [
    "anemia",
    "diabetes",
    "hyperlipidemia",
    "kidney",
    "liver",
    "thyroid"
]

DROP_COLS = [
    "subject_id",
    "hadm_id",
    "charttime"
]

In [5]:
def make_xy(df):

    Y = df[DISEASE_COLS].fillna(0).astype(int)

    X = df.drop(columns=DISEASE_COLS)

    drop_exist = [c for c in DROP_COLS if c in X.columns]
    X = X.drop(columns=drop_exist)

    # gender → numeric
    if "gender" in X.columns:
        X["gender"] = X["gender"].map({"M":1,"F":0})

    
    X = X.select_dtypes(include=[np.number])

    return X, Y

In [6]:
X_train_df, Y_train_df = make_xy(train_df)
X_val_df,   Y_val_df   = make_xy(val_df)
X_test_df,  Y_test_df  = make_xy(test_df)

print(X_train_df.shape)

(93736, 43)


In [7]:
def add_missing_flags(X):

    X = X.copy()

    for c in X.columns:
        X[c+"_missing"] = X[c].isna().astype(int)

    return X

In [8]:
X_train_df = add_missing_flags(X_train_df)
X_val_df   = add_missing_flags(X_val_df)
X_test_df  = add_missing_flags(X_test_df)

In [9]:
print(X_train_df.dtypes)

% Hemoglobin A1c                   float64
Alanine Aminotransferase (ALT)     float64
Albumin                            float64
Alkaline Phosphatase               float64
Asparate Aminotransferase (AST)    float64
                                    ...   
gender_missing                       int64
anchor_age_missing                   int64
anchor_year_missing                  int64
height_cm_missing                    int64
weight_kg_missing                    int64
Length: 86, dtype: object


In [10]:
imputer = SimpleImputer(strategy="median")

X_train = imputer.fit_transform(X_train_df)
X_val   = imputer.transform(X_val_df)
X_test  = imputer.transform(X_test_df)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

Y_train = Y_train_df.values
Y_val   = Y_val_df.values
Y_test  = Y_test_df.values

In [11]:
class TabDataset(Dataset):

    def __init__(self,X,Y):
        self.X = torch.tensor(X,dtype=torch.float32)
        self.Y = torch.tensor(Y,dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self,i):
        return self.X[i],self.Y[i]

In [12]:
none_mask = (Y_train.sum(axis=1)==0)

#focusing on diseased more
weights = np.where(none_mask,0.4,0.6)

sampler = WeightedRandomSampler(
    weights=weights,
    num_samples=len(weights),
    replacement=True
)

In [13]:
train_ds = TabDataset(X_train,Y_train)
val_ds   = TabDataset(X_val,Y_val)
test_ds  = TabDataset(X_test,Y_test)

train_loader = DataLoader(
    train_ds,
    batch_size=256,
    sampler=sampler
)

val_loader = DataLoader(val_ds,batch_size=512)
test_loader = DataLoader(test_ds,batch_size=512)

In [14]:
class MLP(nn.Module):

    def __init__(self,input_dim):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(input_dim,256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256,128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128,6)
        )

    def forward(self,x):
        return self.net(x)

In [15]:
model = MLP(X_train.shape[1]).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

criterion = nn.BCEWithLogitsLoss()

In [16]:
EPOCHS = 20

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for Xb,Yb in train_loader:

        Xb = Xb.to(DEVICE)
        Yb = Yb.to(DEVICE)

        optimizer.zero_grad()

        logits = model(Xb)

        loss = criterion(logits,Yb)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)

        optimizer.step()

        total_loss += loss.item()

    print("Epoch",epoch,"loss",total_loss/len(train_loader))

Epoch 0 loss 0.27499164498794304
Epoch 1 loss 0.23431256466243183
Epoch 2 loss 0.22989869844523697
Epoch 3 loss 0.22718922801985728
Epoch 4 loss 0.22714104168421567
Epoch 5 loss 0.22389378368204882
Epoch 6 loss 0.22543699300419082
Epoch 7 loss 0.22243316150816
Epoch 8 loss 0.2214807801821576
Epoch 9 loss 0.22122583524082923
Epoch 10 loss 0.22024522551236425
Epoch 11 loss 0.21961559765345395
Epoch 12 loss 0.21836346416284994
Epoch 13 loss 0.21892711552677102
Epoch 14 loss 0.2184574242265088
Epoch 15 loss 0.21542095764456393
Epoch 16 loss 0.2167827319709092
Epoch 17 loss 0.21432611224436954
Epoch 18 loss 0.21504208790671273
Epoch 19 loss 0.21504327496488349


In [17]:
def predict(loader):

    model.eval()

    preds = []
    trues = []

    with torch.no_grad():

        for Xb,Yb in loader:

            Xb = Xb.to(DEVICE)

            logits = model(Xb)

            prob = torch.sigmoid(logits).cpu().numpy()

            preds.append(prob)
            trues.append(Yb.numpy())

    preds = np.vstack(preds)
    trues = np.vstack(trues)

    return preds,trues

In [18]:
train_prob,train_true = predict(train_loader)
val_prob,val_true     = predict(val_loader)
test_prob,test_true   = predict(test_loader)

In [19]:
results = []

for i,d in enumerate(DISEASE_COLS):

    t = test_true[:,i]
    p = test_prob[:,i]

    auc = roc_auc_score(t,p)
    pr  = average_precision_score(t,p)

    pred = (p>0.5).astype(int)

    precision = precision_score(t,pred)
    recall    = recall_score(t,pred)
    f1        = f1_score(t,pred)

    results.append({
        "disease":d,
        "AUC":auc,
        "PR_AUC":pr,
        "precision":precision,
        "recall":recall,
        "F1":f1
    })

results_df = pd.DataFrame(results)

results_df

,disease,AUC,PR_AUC,precision,recall,F1
0,anemia,0.936899,0.664331,0.666146,0.574315,0.616831
1,diabetes,0.935509,0.758333,0.721366,0.633272,0.674455
2,hyperlipidemia,0.909807,0.560135,0.622844,0.446672,0.520248
3,kidney,0.964743,0.882900,0.864445,0.755210,0.806144
4,liver,0.920159,0.593798,0.665912,0.437500,0.528065
5,thyroid,0.863719,0.305651,0.614754,0.063131,0.114504


In [20]:
import os
import json
import joblib
import torch

SAVE_DIR = "/kaggle/working/nn_balanced_batch_saved"
os.makedirs(SAVE_DIR, exist_ok=True)

# 1) save model weights
torch.save(model.state_dict(), os.path.join(SAVE_DIR, "model_state_dict.pth"))

# 2) save preprocessing objects
joblib.dump(imputer, os.path.join(SAVE_DIR, "imputer.pkl"))
joblib.dump(scaler, os.path.join(SAVE_DIR, "scaler.pkl"))

# 3) save final feature columns AFTER make_xy + add_missing_flags
joblib.dump(list(X_train_df.columns), os.path.join(SAVE_DIR, "feature_columns.pkl"))

# 4) save labels and config
joblib.dump(DISEASE_COLS, os.path.join(SAVE_DIR, "disease_cols.pkl"))
joblib.dump(DROP_COLS, os.path.join(SAVE_DIR, "drop_cols.pkl"))

# 5) save thresholds (you currently use 0.5 for all)
thresholds = {d: 0.5 for d in DISEASE_COLS}
with open(os.path.join(SAVE_DIR, "thresholds.json"), "w") as f:
    json.dump(thresholds, f, indent=2)

# 6) save metadata
metadata = {
    "input_dim": int(X_train.shape[1]),
    "num_outputs": len(DISEASE_COLS),
    "batch_size_train": 256
}
with open(os.path.join(SAVE_DIR, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved files:")
for fn in os.listdir(SAVE_DIR):
    print("-", fn)

Saved files:
- disease_cols.pkl
- model_state_dict.pth
- imputer.pkl
- feature_columns.pkl
- thresholds.json
- drop_cols.pkl
- metadata.json
- scaler.pkl


In [21]:
import zipfile
import os

zip_path = "/kaggle/working/nn_balanced_batch_saved.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in os.listdir(SAVE_DIR):
        full_path = os.path.join(SAVE_DIR, fn)
        z.write(full_path, arcname=fn)

print("ZIP saved to:", zip_path)

ZIP saved to: /kaggle/working/nn_balanced_batch_saved.zip


## TEST

In [22]:
import os
import json
import joblib
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

LOAD_DIR = "/kaggle/working/nn_balanced_batch_saved"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# load saved preprocessing
imputer = joblib.load(os.path.join(LOAD_DIR, "imputer.pkl"))
scaler = joblib.load(os.path.join(LOAD_DIR, "scaler.pkl"))
feature_columns = joblib.load(os.path.join(LOAD_DIR, "feature_columns.pkl"))
disease_cols = joblib.load(os.path.join(LOAD_DIR, "disease_cols.pkl"))
drop_cols = joblib.load(os.path.join(LOAD_DIR, "drop_cols.pkl"))

with open(os.path.join(LOAD_DIR, "metadata.json"), "r") as f:
    metadata = json.load(f)

with open(os.path.join(LOAD_DIR, "thresholds.json"), "r") as f:
    thresholds = json.load(f)

print("Loaded successfully.")
print("Input dim:", metadata["input_dim"])
print("Diseases:", disease_cols)

Loaded successfully.
Input dim: 86
Diseases: ['anemia', 'diabetes', 'hyperlipidemia', 'kidney', 'liver', 'thyroid']


In [23]:
class MLP(nn.Module):

    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 6)
        )

    def forward(self, x):
        return self.net(x)

model = MLP(metadata["input_dim"]).to(DEVICE)
model.load_state_dict(torch.load(os.path.join(LOAD_DIR, "model_state_dict.pth"), map_location=DEVICE))
model.eval()

print("Model loaded.")

Model loaded.


In [24]:
def make_xy_inference(df, disease_cols, drop_cols):
    df = df.copy()

   
    df = df.drop(columns=[c for c in disease_cols if c in df.columns], errors="ignore")

    # drop cols
    drop_exist = [c for c in drop_cols if c in df.columns]
    df = df.drop(columns=drop_exist, errors="ignore")

    # gender -> numeric
    if "gender" in df.columns:
        df["gender"] = df["gender"].map({"M": 1, "F": 0})

    # keep only numeric
    df = df.select_dtypes(include=[np.number])

    return df


def add_missing_flags_inference(X):
    X = X.copy()

    original_cols = list(X.columns)
    for c in original_cols:
        X[c + "_missing"] = X[c].isna().astype(int)

    return X

In [25]:
def predict_dataframe(df_raw):
    df = df_raw.copy()

    # نفس preprocessing
    X_df = make_xy_inference(df, disease_cols, drop_cols)
    X_df = add_missing_flags_inference(X_df)

    # لازم نفس الأعمدة بالظبط
    for c in feature_columns:
        if c not in X_df.columns:
            X_df[c] = np.nan

    X_df = X_df[feature_columns]

    # impute + scale
    X_np = imputer.transform(X_df)
    X_np = scaler.transform(X_np)

    X_tensor = torch.tensor(X_np, dtype=torch.float32).to(DEVICE)

    with torch.no_grad():
        logits = model(X_tensor)
        probs = torch.sigmoid(logits).cpu().numpy()

    prob_df = pd.DataFrame(probs, columns=disease_cols)

    # apply thresholds
    pred_df = prob_df.copy()
    for d in disease_cols:
        pred_df[d + "_pred"] = (pred_df[d] >= thresholds[d]).astype(int)

    pred_df["none_pred"] = (pred_df[[d + "_pred" for d in disease_cols]].sum(axis=1) == 0).astype(int)

    return pred_df

In [31]:
TEST_PATH = "/kaggle/input/datasets/emanmuhammed/grad-proj/test_multilabel10_group.csv"
test_df = pd.read_csv(TEST_PATH)



In [32]:
 # predictions
pred_df = predict_dataframe(test_df)

# نخلي label names واضحة
pred_df_cols = [c + "_prob" for c in DISEASE_COLS]
pred_df_prob = pred_df[DISEASE_COLS].copy()
pred_df_prob.columns = pred_df_cols

pred_df_labels = pred_df[[d + "_pred" for d in DISEASE_COLS] + ["none_pred"]]

# merge مع original test
result_df = pd.concat([test_df.reset_index(drop=True), pred_df_prob, pred_df_labels], axis=1)

result_df.head()

,subject_id,hadm_id,% Hemoglobin A1c,Alanine Aminotransferase (ALT),Albumin,Alkaline Phosphatase,Asparate Aminotransferase (AST),"Bilirubin, Direct","Bilirubin, Total",Cholesterol Ratio (Total/HDL),...,kidney_prob,liver_prob,thyroid_prob,anemia_pred,diabetes_pred,hyperlipidemia_pred,kidney_pred,liver_pred,thyroid_pred,none_pred
0,10002557,20731670,NaN,247.0,NaN,189.0,246.0,NaN,0.6,NaN,...,0.043359,0.117032,0.096872,0,0,0,0,0,0,1
1,10002804,20769698,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.019307,0.014535,0.008348,0,0,0,0,0,0,1
2,10003400,26090619,NaN,28.0,2.7,83.0,24.0,NaN,0.4,NaN,...,0.171447,0.026402,0.144169,0,0,0,0,0,0,1
3,10003412,28884815,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.014749,0.001852,0.005151,0,0,0,0,0,0,1
4,10004113,29879900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000082,0.000009,0.000013,0,0,0,0,0,0,1


In [34]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

metrics = []

for d in DISEASE_COLS:
    y_true = test_df[d]
    y_pred = result_df[d + "_pred"]

    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)

    metrics.append([d, acc, prec, rec, f1])

metrics_df = pd.DataFrame(metrics, columns=["disease", "accuracy", "precision", "recall", "f1"])

metrics_df

,disease,accuracy,precision,recall,f1
0,anemia,0.921083,0.666146,0.574315,0.616831
1,diabetes,0.916961,0.721366,0.633272,0.674455
2,hyperlipidemia,0.904097,0.622844,0.446672,0.520248
3,kidney,0.945468,0.864445,0.755210,0.806144
4,liver,0.947802,0.665912,0.437500,0.528065
5,thyroid,0.942389,0.614754,0.063131,0.114504


In [35]:
total_correct = 0
total_all = 0

for d in DISEASE_COLS:
    total_correct += (test_df[d] == result_df[d + "_pred"]).sum()
    total_all += len(test_df)

overall_accuracy = total_correct / total_all

print("Overall accuracy:", overall_accuracy)

Overall accuracy: 0.9296333085009519


In [36]:
exact_match = (test_df[DISEASE_COLS].values == result_df[[d + "_pred" for d in DISEASE_COLS]].values).all(axis=1).mean()

print("Exact match accuracy:", exact_match)

Exact match accuracy: 0.7703998013409485


In [37]:
SAVE_PATH = "/kaggle/working/test_predictions_full.csv"
result_df.to_csv(SAVE_PATH, index=False)

print("Saved to:", SAVE_PATH)

Saved to: /kaggle/working/test_predictions_full.csv


In [38]:
for d in DISEASE_COLS:
    result_df[d + "_correct"] = (test_df[d] == result_df[d + "_pred"]).astype(int)

result_df.head()

,subject_id,hadm_id,% Hemoglobin A1c,Alanine Aminotransferase (ALT),Albumin,Alkaline Phosphatase,Asparate Aminotransferase (AST),"Bilirubin, Direct","Bilirubin, Total",Cholesterol Ratio (Total/HDL),...,kidney_pred,liver_pred,thyroid_pred,none_pred,anemia_correct,diabetes_correct,hyperlipidemia_correct,kidney_correct,liver_correct,thyroid_correct
0,10002557,20731670,NaN,247.0,NaN,189.0,246.0,NaN,0.6,NaN,...,0,0,0,1,1,1,1,1,1,1
1,10002804,20769698,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,1,1,1,1,1,1,1
2,10003400,26090619,NaN,28.0,2.7,83.0,24.0,NaN,0.4,NaN,...,0,0,0,1,1,1,1,1,1,1
3,10003412,28884815,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,1,1,1,1,1,1,1
4,10004113,29879900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,1,1,1,1,1,1,1
